In [1]:
#Tsfnet corrected - subject dependent
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # نسخه ارتقایافته: Transformer block کامل با FFN
    # مقاله TSFNet: CAFE تنها ~2.77M پارامتر دارد (Table 6: 3.33M کل - 0.57M بدون CAFE = 2.77M).
    # نسخه قبلی فقط MultiheadAttention(64,nhead=4) بود که خیلی سبک‌تر از مقاله است.
    # این نسخه:
    #   1. d_model را به 256 افزایش می‌دهد (از 64) — اصلی‌ترین عامل افزایش پارامتر
    #   2. FFN با dim_feedforward=512 اضافه می‌کند — مثل Transformer استاندارد
    #   3. nhead=8 به جای 4 — مقاله "multi-head attention" را تأکید می‌کند
    # پارامترهای هر CrossAttention block با d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ≈ 262K
    #   FFN: 256*512 + 512*256 ≈ 262K
    #   LayerNorm × 2: ~1K
    #   مجموع هر block: ~525K — دو block (cafe_attn + cafe_attn2) ≈ 1.05M
    #   با projection layers (nirs_seq_proj, gate layers): ≈ 1.3-1.5M برای کل CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN — مثل Transformer encoder block استاندارد
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention با residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN با residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # مقاله یک شاخه Conv3d مستقل دارد که EEG tensor را می‌گیرد اما وزن‌هایش
        # کاملاً جداگانه از EEG branch یاد می‌گیرند. هدف این شاخه آماده‌سازی
        # نمایش EEG برای ترکیب با fNIRS است، نه classification خالص EEG.
        # پارامترهای kernel/stride دقیقاً مثل EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # خروجی پس از GAP روی محور زمانی: F_fusion ∈ R^{32×4×4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        # مقاله: attention map از روی feature VOLUME کامل (با بعد زمانی) ساخته می‌شود:
        #   Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # Conv3d روی volume کامل اعمال می‌شود، سپس GAP بعد زمانی را حذف می‌کند.
        # نتیجه: attention map فضایی (h,w) که اطلاعات زمانی را در خود جذب کرده.
        #
        # EFGF1 — روی feature volume پس از Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) → خروجی (batch,1,8,8,~95)، سپس GAP زمانی
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 — روی feature volume پس از Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) → خروجی (batch,1,4,4,~48)، سپس GAP زمانی
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # projection نهایی F_fusion (spatial 32×4×4) به 64-d برای classifier
        # CAFE روی spatial map کار می‌کند، پس flatten فقط برای fc_fusion است
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE — d_model=128
        # CHANGED: از 256 به 128 کاهش یافت.
        # با 26 subject و cross-subject scenario، 1.5M پارامتر CAFE باعث overfit شدید شد
        # (train acc به 99% رسید در حالی که val acc در 54% ماند).
        # 128-d حد وسط مناسبی است: بیشتر از 64-d اولیه (capacity بیشتر)، کمتر از 256-d (overfit کمتر).
        # پارامتر CAFE با d_model=128: ~350K — در مقابل ~1.39M برای 256-d.
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers (مقاله Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: hidden layer بین feature (64-d) و output (3 کلاس)
        # از direct mapping بهتر عمل می‌کند.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # مقاله Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # قبلاً: F.softmax(stack(w1,w2,w3)) که مجموع وزن‌ها را اجباراً 1 می‌کند و یک constraint
        # اضافه ایجاد می‌کند که در مقاله وجود ندارد.
        # اکنون: torch.sigmoid(wi) برای هر وزن به صورت مستقل، دقیقاً مطابق Eq.4.
        # sigmoid ∈ (0,1) بنابراین هر وزن بین 0 و 1 است اما مجموع آن‌ها ممکن است ≠ 1 باشد.
        # مقاله: "Mean denotes the averaging operation" → Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # یعنی میانگین ساده (نه weighted sum normalized) از سه پیش‌بینی وزن‌دار.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # GAP زمانی فقط برای classifier EEG (نه برای EFGF)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 هنوز محاسبه نمی‌شود — باید بعد از EFGF1 (fx1_updated) محاسبه شود
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود، نه fx1 اصلی

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP روی segment و زمان برای fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        # مقاله Eq.2: Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold برای اعمال مستقل روی هر segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) — mean روی segments
        # Conv3d روی volume کامل (بعد زمانی حفظ شده) → GAP زمانی → attention map فضایی
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') — T' کوچک‌شده
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') — GAP زمانی
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = γ1·F^eeg_1 + (1-γ1)·F_fus1 + Phi1⊗F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') — F̂_fusion خروجی EFGF1

        # ── Fusion branch — Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) ──
        # قبلاً: fx2 = fus_conv2(fx1)  ← EFGF1 هیچ اثری روی شاخه fusion نداشت
        # اکنون: fx2 = fus_conv2(fx1_updated)  ← مطابق مقاله Figure 4
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود.
        # این تضمین می‌کند که EFGF1 واقعاً در زنجیره پردازش fusion branch اثر دارد
        # و gradient از Conv2 به EFGF1 جاری می‌شود — در نسخه قبلی این ارتباط قطع بود.
        # FIX: fx1_updated ممکن است spatial dims کوچکتر از fx1 داشته باشد (به دلیل
        # padding truncation در EFGF1). adaptive_avg_pool3d برای align کردن استفاده می‌شود
        # تا fus_conv2 با in_channels=16 و kernel صحیح کار کند.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) — همان shape fx1 اصلی
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 → fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) — حجم کامل برای EFGF2 حفظ می‌شود

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') — attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # GAP زمانی پس از EFGF2 → spatial feature map برای CAFE و classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' ممکن است بعد از Conv3d EFGF کوچک‌تر از 4×4 شده باشند
        # adaptive pool برای تطابق با fusion_proj که انتظار 32×4×4 دارد
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # برای L_EFGF (Eq.12): PCC بین eeg_feat_map و Phi2 (هم‌اندازه‌شده)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # اگر spatial dims با eeg_feat_map فرق دارد، به هم align می‌کنیم
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion ← fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS ← fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # بازگشت به 64-d برای classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # قبلاً: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax مجموع وزن‌ها را اجباراً 1 می‌کند — این در مقاله وجود ندارد.
        # مقاله Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # یعنی: هر وزن مستقلاً از sigmoid می‌گذرد، سپس میانگین وزن‌دار گرفته می‌شود.
        # اکنون: sigmoid برای هر وزن، Mean برای ترکیب — دقیقاً Eq.4.
        w1 = torch.sigmoid(self.w1)  # FIX: sigmoid به جای softmax — مطابق Eq.4
        w2 = torch.sigmoid(self.w2)  # FIX
        w3 = torch.sigmoid(self.w3)  # FIX
        final_out = (w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out) / 3.0  # FIX: Mean مطابق Eq.4

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]  # مقاله: Adam بدون weight decay (مقدار پیش‌فرض = 0)

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

# TWO-STAGE TRAINING (paper Sec. 3.6):
# Stage 1: train on 80% of train set, early-stop on 20% val, patience=50, max=300 epochs.
#          Best checkpoint (highest val acc) is restored at end of stage 1.
# Stage 2: retrain on FULL train set (no val split), stop when train loss drops below
#          the best stage-1 train loss achieved, max=200 epochs.
STAGE1_MAX_EPOCHS = 300   # paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # paper says patience 50
STAGE2_MAX_EPOCHS = 200   # paper Sec. 3.6 says max 200 for stage 2

# FIX (مقاله Sec. 3.6 — Paper-Compliant Fix 3/3: Window-level 4:1 val split):
# قبلاً: subject-level split — 2 subject کامل برای val نگه داشته می‌شد.
# مقاله Sec. 3.6: "the training set was randomly split into a first-stage training subset
# and a validation set in a 4:1 ratio."
# این split روی پنجره‌های زمانی (windows) است، نه روی subjects.
# اکنون: TRAIN_SUBS کامل برای stage-1 استفاده می‌شود، اما 20% از windows به صورت
# تصادفی به عنوان val جدا می‌شوند — دقیقاً مطابق روش مقاله.
# نکته: S1_VAL_SUBS و S1_TRAIN_SUBS دیگر استفاده نمی‌شوند؛ split درون run_training
# با SubsetRandomSampler روی indices انجام می‌شود.
# این تغییر باعث می‌شود val acc از windows همان subjects train بیاید (مثل مقاله).


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device):  # NEW
    print("\n" + "#" * 70)
    print(f"# WEIGHT_DECAY = {weight_decay}")
    print("#" * 70)

    # FIX (Paper-Compliant Fix 3/3): window-level 4:1 split مطابق مقاله Sec. 3.6
    # MMapHybridDataset کل TRAIN_SUBS را بارگذاری می‌کند، سپس indices به صورت تصادفی
    # به نسبت 4:1 تقسیم می‌شوند (train_indices: 80%، val_indices: 20%).
    # SubsetRandomSampler این indices را به DataLoader می‌دهد تا split واقعی باشد.
    from torch.utils.data import SubsetRandomSampler  # FIX: window-level split

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    test_dataset  = MMapHybridDataset(TEST_SUBS,  "Test_Set")

    # FIX: window-level 4:1 split روی TRAIN_SUBS — مطابق مقاله Sec. 3.6
    n_total_train = len(train_dataset)
    indices = list(range(n_total_train))
    random.shuffle(indices)  # seed=42 قبلاً set شده — reproducible
    split = int(n_total_train * 0.8)  # 80% train, 20% val — نسبت 4:1 مقاله
    s1_train_indices = indices[:split]
    s1_val_indices   = indices[split:]
    print(f"[Paper-Compliant] Window-level 4:1 split: "
          f"{len(s1_train_indices)} train windows, {len(s1_val_indices)} val windows "
          f"(from same {len(TRAIN_SUBS)} subjects, ratio={len(s1_train_indices)/len(s1_val_indices):.1f}:1)")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 80% windows
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 20% windows
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer (stage 1) -- stage 2 creates its own optimizer2 below
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # label_smoothing=0.1: Müller et al. (2019) NeurIPS "When Does Label Smoothing Help?"
    # در cross-subject، train/test distribution فرق دارند — label smoothing calibration را بهبود می‌دهد
    # و مانع از overconfident predictions می‌شود که مستقیماً به overfitting منجر می‌شود.
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_wd{weight_decay}.pth')

    # ------------------------------------------------------------------ #
    # STAGE 1: train on 80% of TRAIN windows, early-stop on 20% val      #
    # مطابق مقاله Sec. 3.6: window-level 4:1 split                       #
    # ------------------------------------------------------------------ #
    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of TRAIN_SUBS), "
          f"val on {len(s1_val_indices)} windows (20% of TRAIN_SUBS) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | window-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')  # loss در epoch بهترین val
    stage1_final_train_loss    = float('inf')  # loss آخرین epoch — fallback وقتی best val در epoch 1 است
    train_acc = 0.0

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss  # هر epoch آپدیت می‌شود

        # stage-1 validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    # Restore best stage-1 weights before stage 2
    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(TRAIN_SUBS)} subjects) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    # Fresh optimizer for stage 2
    optimizer2 = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=weight_decay)
    scheduler2  = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=5)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)
        scheduler2.step(avg_s2_loss)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        # Reached max epochs without meeting loss criterion -- save anyway
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (weight_decay={weight_decay})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    # FIX: learned weights اکنون از sigmoid می‌آیند (نه softmax)
    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap = (train_acc * 100) - test_acc
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'weight_decay': weight_decay,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_wd{weight_decay}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- runs the weight-decay grid search
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Weight decay grid to test: {WEIGHT_DECAY_GRID}")

    all_grid_results = []
    for wd in WEIGHT_DECAY_GRID:
        result = run_training(wd, device)
        all_grid_results.append(result)

    print("\n" + "=" * 70)
    print("WEIGHT DECAY GRID SEARCH SUMMARY")
    print("=" * 70)
    print(f"{'weight_decay':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'overfit_gap':>12}")
    for r in all_grid_results:
        print(f"{r['weight_decay']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['overfitting_gap']:>+11.2f}pp")

    best_by_test_acc = max(all_grid_results, key=lambda r: r['test_accuracy_final'])
    best_by_gap = min(all_grid_results, key=lambda r: r['overfitting_gap'])
    print(f"\nBest weight_decay by test accuracy: {best_by_test_acc['weight_decay']} "
          f"({best_by_test_acc['test_accuracy_final']:.2f}%)")
    print(f"Best weight_decay by smallest overfitting gap: {best_by_gap['weight_decay']} "
          f"({best_by_gap['overfitting_gap']:+.2f}pp)")

    summary_path = os.path.join(PROCESSED_DATA_PATH, 'weight_decay_grid_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(all_grid_results, f, indent=2)
    print(f"\nFull grid-search summary saved to: {summary_path}")

    print("\nCleanup completed. Temporary disk files removed.")

Running on Device: cuda
Weight decay grid to test: [0.01, 0.02, 0.05]

######################################################################
# WEIGHT_DECAY = 0.01
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Test_Set - 5 subjects on Drive D...
VP008 veri

In [2]:
#version2
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # نسخه ارتقایافته: Transformer block کامل با FFN
    # مقاله TSFNet: CAFE تنها ~2.77M پارامتر دارد (Table 6: 3.33M کل - 0.57M بدون CAFE = 2.77M).
    # نسخه قبلی فقط MultiheadAttention(64,nhead=4) بود که خیلی سبک‌تر از مقاله است.
    # این نسخه:
    #   1. d_model را به 256 افزایش می‌دهد (از 64) — اصلی‌ترین عامل افزایش پارامتر
    #   2. FFN با dim_feedforward=512 اضافه می‌کند — مثل Transformer استاندارد
    #   3. nhead=8 به جای 4 — مقاله "multi-head attention" را تأکید می‌کند
    # پارامترهای هر CrossAttention block با d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ≈ 262K
    #   FFN: 256*512 + 512*256 ≈ 262K
    #   LayerNorm × 2: ~1K
    #   مجموع هر block: ~525K — دو block (cafe_attn + cafe_attn2) ≈ 1.05M
    #   با projection layers (nirs_seq_proj, gate layers): ≈ 1.3-1.5M برای کل CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN — مثل Transformer encoder block استاندارد
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention با residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN با residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # مقاله یک شاخه Conv3d مستقل دارد که EEG tensor را می‌گیرد اما وزن‌هایش
        # کاملاً جداگانه از EEG branch یاد می‌گیرند. هدف این شاخه آماده‌سازی
        # نمایش EEG برای ترکیب با fNIRS است، نه classification خالص EEG.
        # پارامترهای kernel/stride دقیقاً مثل EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # خروجی پس از GAP روی محور زمانی: F_fusion ∈ R^{32×4×4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        # مقاله: attention map از روی feature VOLUME کامل (با بعد زمانی) ساخته می‌شود:
        #   Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # Conv3d روی volume کامل اعمال می‌شود، سپس GAP بعد زمانی را حذف می‌کند.
        # نتیجه: attention map فضایی (h,w) که اطلاعات زمانی را در خود جذب کرده.
        #
        # EFGF1 — روی feature volume پس از Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) → خروجی (batch,1,8,8,~95)، سپس GAP زمانی
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 — روی feature volume پس از Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) → خروجی (batch,1,4,4,~48)، سپس GAP زمانی
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # projection نهایی F_fusion (spatial 32×4×4) به 64-d برای classifier
        # CAFE روی spatial map کار می‌کند، پس flatten فقط برای fc_fusion است
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE — d_model=128
        # CHANGED: از 256 به 128 کاهش یافت.
        # با 26 subject و cross-subject scenario، 1.5M پارامتر CAFE باعث overfit شدید شد
        # (train acc به 99% رسید در حالی که val acc در 54% ماند).
        # 128-d حد وسط مناسبی است: بیشتر از 64-d اولیه (capacity بیشتر)، کمتر از 256-d (overfit کمتر).
        # پارامتر CAFE با d_model=128: ~350K — در مقابل ~1.39M برای 256-d.
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers (مقاله Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: hidden layer بین feature (64-d) و output (3 کلاس)
        # از direct mapping بهتر عمل می‌کند.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # مقاله Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # قبلاً: F.softmax(stack(w1,w2,w3)) که مجموع وزن‌ها را اجباراً 1 می‌کند و یک constraint
        # اضافه ایجاد می‌کند که در مقاله وجود ندارد.
        # اکنون: torch.sigmoid(wi) برای هر وزن به صورت مستقل، دقیقاً مطابق Eq.4.
        # sigmoid ∈ (0,1) بنابراین هر وزن بین 0 و 1 است اما مجموع آن‌ها ممکن است ≠ 1 باشد.
        # مقاله: "Mean denotes the averaging operation" → Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # یعنی میانگین ساده (نه weighted sum normalized) از سه پیش‌بینی وزن‌دار.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # GAP زمانی فقط برای classifier EEG (نه برای EFGF)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 هنوز محاسبه نمی‌شود — باید بعد از EFGF1 (fx1_updated) محاسبه شود
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود، نه fx1 اصلی

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP روی segment و زمان برای fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        # مقاله Eq.2: Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold برای اعمال مستقل روی هر segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) — mean روی segments
        # Conv3d روی volume کامل (بعد زمانی حفظ شده) → GAP زمانی → attention map فضایی
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') — T' کوچک‌شده
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') — GAP زمانی
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = γ1·F^eeg_1 + (1-γ1)·F_fus1 + Phi1⊗F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') — F̂_fusion خروجی EFGF1

        # ── Fusion branch — Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) ──
        # قبلاً: fx2 = fus_conv2(fx1)  ← EFGF1 هیچ اثری روی شاخه fusion نداشت
        # اکنون: fx2 = fus_conv2(fx1_updated)  ← مطابق مقاله Figure 4
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود.
        # این تضمین می‌کند که EFGF1 واقعاً در زنجیره پردازش fusion branch اثر دارد
        # و gradient از Conv2 به EFGF1 جاری می‌شود — در نسخه قبلی این ارتباط قطع بود.
        # FIX: fx1_updated ممکن است spatial dims کوچکتر از fx1 داشته باشد (به دلیل
        # padding truncation در EFGF1). adaptive_avg_pool3d برای align کردن استفاده می‌شود
        # تا fus_conv2 با in_channels=16 و kernel صحیح کار کند.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) — همان shape fx1 اصلی
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 → fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) — حجم کامل برای EFGF2 حفظ می‌شود

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') — attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # GAP زمانی پس از EFGF2 → spatial feature map برای CAFE و classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' ممکن است بعد از Conv3d EFGF کوچک‌تر از 4×4 شده باشند
        # adaptive pool برای تطابق با fusion_proj که انتظار 32×4×4 دارد
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # برای L_EFGF (Eq.12): PCC بین eeg_feat_map و Phi2 (هم‌اندازه‌شده)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # اگر spatial dims با eeg_feat_map فرق دارد، به هم align می‌کنیم
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion ← fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS ← fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # بازگشت به 64-d برای classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # قبلاً: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax مجموع وزن‌ها را اجباراً 1 می‌کند — این در مقاله وجود ندارد.
        # مقاله Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # یعنی: هر وزن مستقلاً از sigmoid می‌گذرد، سپس میانگین وزن‌دار گرفته می‌شود.
        # اکنون: sigmoid برای هر وزن، Mean برای ترکیب — دقیقاً Eq.4.
        w1 = torch.sigmoid(self.w1)  # sigmoid مطابق Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        # FIX (3-class adaptation): حذف /3 — تقسیم بر 3 logit‌ها را scale می‌کرد و
        # gradient را 3 برابر کوچک می‌کرد (gradient از w*logit → gradient/3)، که
        # یادگیری را کند و ناپایدار می‌کرد. روی argmax بی‌اثر بود اما روی loss
        # مصنوعاً مقدار را بالا نگه می‌داشت (آزمون: loss بدون /3 = 0.817 ، با /3 = 0.933).
        # weighted sum مستقیم هم از نظر ریاضی صحیح‌تر است: CrossEntropyLoss روی
        # logit کار می‌کند و نیازی به normalization قبلی ندارد.
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out  # FIX: /3 حذف شد

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [ 0.02 ]  # مقاله: Adam بدون weight decay (مقدار پیش‌فرض = 0)

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

# TWO-STAGE TRAINING (paper Sec. 3.6):
# Stage 1: train on 80% of train set, early-stop on 20% val, patience=50, max=300 epochs.
#          Best checkpoint (highest val acc) is restored at end of stage 1.
# Stage 2: retrain on FULL train set (no val split), stop when train loss drops below
#          the best stage-1 train loss achieved, max=200 epochs.
STAGE1_MAX_EPOCHS = 300   # paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # paper says patience 50
STAGE2_MAX_EPOCHS = 200   # paper Sec. 3.6 says max 200 for stage 2

# FIX (مقاله Sec. 3.6 — Paper-Compliant Fix 3/3: Window-level 4:1 val split):
# قبلاً: subject-level split — 2 subject کامل برای val نگه داشته می‌شد.
# مقاله Sec. 3.6: "the training set was randomly split into a first-stage training subset
# and a validation set in a 4:1 ratio."
# این split روی پنجره‌های زمانی (windows) است، نه روی subjects.
# اکنون: TRAIN_SUBS کامل برای stage-1 استفاده می‌شود، اما 20% از windows به صورت
# تصادفی به عنوان val جدا می‌شوند — دقیقاً مطابق روش مقاله.
# نکته: S1_VAL_SUBS و S1_TRAIN_SUBS دیگر استفاده نمی‌شوند؛ split درون run_training
# با SubsetRandomSampler روی indices انجام می‌شود.
# این تغییر باعث می‌شود val acc از windows همان subjects train بیاید (مثل مقاله).


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device):  # NEW
    print("\n" + "#" * 70)
    print(f"# WEIGHT_DECAY = {weight_decay}")
    print("#" * 70)

    # FIX (Paper-Compliant Fix 3/3): window-level 4:1 split مطابق مقاله Sec. 3.6
    # MMapHybridDataset کل TRAIN_SUBS را بارگذاری می‌کند، سپس indices به صورت تصادفی
    # به نسبت 4:1 تقسیم می‌شوند (train_indices: 80%، val_indices: 20%).
    # SubsetRandomSampler این indices را به DataLoader می‌دهد تا split واقعی باشد.
    from torch.utils.data import SubsetRandomSampler  # FIX: window-level split

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    test_dataset  = MMapHybridDataset(TEST_SUBS,  "Test_Set")

    # FIX: window-level 4:1 split روی TRAIN_SUBS — مطابق مقاله Sec. 3.6
    n_total_train = len(train_dataset)
    indices = list(range(n_total_train))
    random.shuffle(indices)  # seed=42 قبلاً set شده — reproducible
    split = int(n_total_train * 0.8)  # 80% train, 20% val — نسبت 4:1 مقاله
    s1_train_indices = indices[:split]
    s1_val_indices   = indices[split:]
    print(f"[Paper-Compliant] Window-level 4:1 split: "
          f"{len(s1_train_indices)} train windows, {len(s1_val_indices)} val windows "
          f"(from same {len(TRAIN_SUBS)} subjects, ratio={len(s1_train_indices)/len(s1_val_indices):.1f}:1)")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 80% windows
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 20% windows
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer (stage 1) -- stage 2 creates its own optimizer2 below
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # FIX (3-class adaptation): class-weighted CrossEntropyLoss برای n-back imbalance
    # مقاله روی تکالیف binary (balanced) کار می‌کرد — label_smoothing به تنهایی کافی بود.
    # در n-back سه‌کلاسه، 0-back معمولاً windows بیشتری دارد (آسان‌تر، طولانی‌تر)،
    # که بدون تصحیح باعث می‌شود model به سمت کلاس majority bias پیدا کند و
    # 2-back و 3-back را کمتر یاد بگیرد — دقیقاً مشکلی که در BCI cross-subject
    # classification گزارش شده (Lotte et al., 2018).
    # راه‌حل: inverse-frequency weighting — کلاس‌های کم‌نمونه‌تر وزن بیشتری می‌گیرند.
    # وزن‌ها از train_dataset.y محاسبه می‌شوند (نه test) تا از data leakage جلوگیری شود.
    # label_smoothing=0.1 همچنان حفظ می‌شود — هر دو با هم کار می‌کنند:
    #   class_weight: bias مدل به کلاس majority را اصلاح می‌کند
    #   label_smoothing: overconfidence را کاهش می‌دهد
    # منبع: King & Zeng (2001) — inverse frequency weighting برای imbalanced classification
    _train_labels   = train_dataset.y                                        # numpy array از همه train windows
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float) # [n_0back, n_2back, n_3back]
    _class_weights  = 1.0 / (_class_counts + 1e-6)                          # inverse frequency
    _class_weights  = _class_weights / _class_weights.sum() * 3             # normalize: mean weight = 1
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    # label_smoothing=0.1: Müller et al. (2019) NeurIPS "When Does Label Smoothing Help?"
    # در cross-subject، train/test distribution فرق دارند — label smoothing calibration را بهبود می‌دهد
    # و مانع از overconfident predictions می‌شود که مستقیماً به overfitting منجر می‌شود.
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor, label_smoothing=0.1)  # FIX: class weights اضافه شد

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_wd{weight_decay}.pth')

    # ------------------------------------------------------------------ #
    # STAGE 1: train on 80% of TRAIN windows, early-stop on 20% val      #
    # مطابق مقاله Sec. 3.6: window-level 4:1 split                       #
    # ------------------------------------------------------------------ #
    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of TRAIN_SUBS), "
          f"val on {len(s1_val_indices)} windows (20% of TRAIN_SUBS) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | window-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')  # loss در epoch بهترین val
    stage1_final_train_loss    = float('inf')  # loss آخرین epoch — fallback وقتی best val در epoch 1 است
    train_acc = 0.0

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss  # هر epoch آپدیت می‌شود

        # stage-1 validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    # Restore best stage-1 weights before stage 2
    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(TRAIN_SUBS)} subjects) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    # Fresh optimizer for stage 2
    optimizer2 = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=weight_decay)
    scheduler2  = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=5)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)
        scheduler2.step(avg_s2_loss)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        # Reached max epochs without meeting loss criterion -- save anyway
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (weight_decay={weight_decay})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    # FIX: learned weights اکنون از sigmoid می‌آیند (نه softmax)
    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap = (train_acc * 100) - test_acc
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'weight_decay': weight_decay,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_wd{weight_decay}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- runs the weight-decay grid search
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Weight decay grid to test: {WEIGHT_DECAY_GRID}")

    all_grid_results = []
    for wd in WEIGHT_DECAY_GRID:
        result = run_training(wd, device)
        all_grid_results.append(result)

    print("\n" + "=" * 70)
    print("WEIGHT DECAY GRID SEARCH SUMMARY")
    print("=" * 70)
    print(f"{'weight_decay':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'overfit_gap':>12}")
    for r in all_grid_results:
        print(f"{r['weight_decay']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['overfitting_gap']:>+11.2f}pp")

    best_by_test_acc = max(all_grid_results, key=lambda r: r['test_accuracy_final'])
    best_by_gap = min(all_grid_results, key=lambda r: r['overfitting_gap'])
    print(f"\nBest weight_decay by test accuracy: {best_by_test_acc['weight_decay']} "
          f"({best_by_test_acc['test_accuracy_final']:.2f}%)")
    print(f"Best weight_decay by smallest overfitting gap: {best_by_gap['weight_decay']} "
          f"({best_by_gap['overfitting_gap']:+.2f}pp)")

    summary_path = os.path.join(PROCESSED_DATA_PATH, 'weight_decay_grid_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(all_grid_results, f, indent=2)
    print(f"\nFull grid-search summary saved to: {summary_path}")

    print("\nCleanup completed. Temporary disk files removed.")


Running on Device: cuda
Weight decay grid to test: [0.02]

######################################################################
# WEIGHT_DECAY = 0.02
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Test_Set - 5 subjects on Drive D...
VP008 verified: 1023 w

In [1]:
#Tsfnet cross session
#version3
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (مطابق مقاله TSFNet Sec. 3.6):
# به جای تقسیم subjects به train/test (cross-subject)، برای هر subject
# یک session به عنوان test و دو session به عنوان train استفاده می‌شود.
# Hold-Out: 3 بار اجرا — هر بار یک session مختلف به عنوان test.
# نتیجه نهایی = میانگین test accuracy سه run.
# این رویکرد دقیقاً مطابق Sec. 3.6 مقاله است که cross-session را ارزیابی می‌کند.
# منبع: VPxxx_session_ids.npy که توسط generate_session_ids.py تولید شده،
# برای هر window مقدار 1، 2 یا 3 (session number) را نگه می‌دارد.

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: فیلتر بر اساس session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: اگر session_mask وجود دارد، فقط windows مربوطه را load کن
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # نسخه ارتقایافته: Transformer block کامل با FFN
    # مقاله TSFNet: CAFE تنها ~2.77M پارامتر دارد (Table 6: 3.33M کل - 0.57M بدون CAFE = 2.77M).
    # نسخه قبلی فقط MultiheadAttention(64,nhead=4) بود که خیلی سبک‌تر از مقاله است.
    # این نسخه:
    #   1. d_model را به 256 افزایش می‌دهد (از 64) — اصلی‌ترین عامل افزایش پارامتر
    #   2. FFN با dim_feedforward=512 اضافه می‌کند — مثل Transformer استاندارد
    #   3. nhead=8 به جای 4 — مقاله "multi-head attention" را تأکید می‌کند
    # پارامترهای هر CrossAttention block با d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ≈ 262K
    #   FFN: 256*512 + 512*256 ≈ 262K
    #   LayerNorm × 2: ~1K
    #   مجموع هر block: ~525K — دو block (cafe_attn + cafe_attn2) ≈ 1.05M
    #   با projection layers (nirs_seq_proj, gate layers): ≈ 1.3-1.5M برای کل CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN — مثل Transformer encoder block استاندارد
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention با residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN با residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # مقاله یک شاخه Conv3d مستقل دارد که EEG tensor را می‌گیرد اما وزن‌هایش
        # کاملاً جداگانه از EEG branch یاد می‌گیرند. هدف این شاخه آماده‌سازی
        # نمایش EEG برای ترکیب با fNIRS است، نه classification خالص EEG.
        # پارامترهای kernel/stride دقیقاً مثل EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # خروجی پس از GAP روی محور زمانی: F_fusion ∈ R^{32×4×4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        # مقاله: attention map از روی feature VOLUME کامل (با بعد زمانی) ساخته می‌شود:
        #   Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # Conv3d روی volume کامل اعمال می‌شود، سپس GAP بعد زمانی را حذف می‌کند.
        # نتیجه: attention map فضایی (h,w) که اطلاعات زمانی را در خود جذب کرده.
        #
        # EFGF1 — روی feature volume پس از Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) → خروجی (batch,1,8,8,~95)، سپس GAP زمانی
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 — روی feature volume پس از Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) → خروجی (batch,1,4,4,~48)، سپس GAP زمانی
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # projection نهایی F_fusion (spatial 32×4×4) به 64-d برای classifier
        # CAFE روی spatial map کار می‌کند، پس flatten فقط برای fc_fusion است
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE — d_model=128
        # CHANGED: از 256 به 128 کاهش یافت.
        # با 26 subject و cross-subject scenario، 1.5M پارامتر CAFE باعث overfit شدید شد
        # (train acc به 99% رسید در حالی که val acc در 54% ماند).
        # 128-d حد وسط مناسبی است: بیشتر از 64-d اولیه (capacity بیشتر)، کمتر از 256-d (overfit کمتر).
        # پارامتر CAFE با d_model=128: ~350K — در مقابل ~1.39M برای 256-d.
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers (مقاله Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: hidden layer بین feature (64-d) و output (3 کلاس)
        # از direct mapping بهتر عمل می‌کند.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # مقاله Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # قبلاً: F.softmax(stack(w1,w2,w3)) که مجموع وزن‌ها را اجباراً 1 می‌کند و یک constraint
        # اضافه ایجاد می‌کند که در مقاله وجود ندارد.
        # اکنون: torch.sigmoid(wi) برای هر وزن به صورت مستقل، دقیقاً مطابق Eq.4.
        # sigmoid ∈ (0,1) بنابراین هر وزن بین 0 و 1 است اما مجموع آن‌ها ممکن است ≠ 1 باشد.
        # مقاله: "Mean denotes the averaging operation" → Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # یعنی میانگین ساده (نه weighted sum normalized) از سه پیش‌بینی وزن‌دار.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # GAP زمانی فقط برای classifier EEG (نه برای EFGF)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 هنوز محاسبه نمی‌شود — باید بعد از EFGF1 (fx1_updated) محاسبه شود
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود، نه fx1 اصلی

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP روی segment و زمان برای fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        # مقاله Eq.2: Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold برای اعمال مستقل روی هر segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) — mean روی segments
        # Conv3d روی volume کامل (بعد زمانی حفظ شده) → GAP زمانی → attention map فضایی
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') — T' کوچک‌شده
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') — GAP زمانی
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = γ1·F^eeg_1 + (1-γ1)·F_fus1 + Phi1⊗F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') — F̂_fusion خروجی EFGF1

        # ── Fusion branch — Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) ──
        # قبلاً: fx2 = fus_conv2(fx1)  ← EFGF1 هیچ اثری روی شاخه fusion نداشت
        # اکنون: fx2 = fus_conv2(fx1_updated)  ← مطابق مقاله Figure 4
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود.
        # این تضمین می‌کند که EFGF1 واقعاً در زنجیره پردازش fusion branch اثر دارد
        # و gradient از Conv2 به EFGF1 جاری می‌شود — در نسخه قبلی این ارتباط قطع بود.
        # FIX: fx1_updated ممکن است spatial dims کوچکتر از fx1 داشته باشد (به دلیل
        # padding truncation در EFGF1). adaptive_avg_pool3d برای align کردن استفاده می‌شود
        # تا fus_conv2 با in_channels=16 و kernel صحیح کار کند.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) — همان shape fx1 اصلی
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 → fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) — حجم کامل برای EFGF2 حفظ می‌شود

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') — attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # GAP زمانی پس از EFGF2 → spatial feature map برای CAFE و classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' ممکن است بعد از Conv3d EFGF کوچک‌تر از 4×4 شده باشند
        # adaptive pool برای تطابق با fusion_proj که انتظار 32×4×4 دارد
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # برای L_EFGF (Eq.12): PCC بین eeg_feat_map و Phi2 (هم‌اندازه‌شده)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # اگر spatial dims با eeg_feat_map فرق دارد، به هم align می‌کنیم
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion ← fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS ← fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # بازگشت به 64-d برای classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # قبلاً: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax مجموع وزن‌ها را اجباراً 1 می‌کند — این در مقاله وجود ندارد.
        # مقاله Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # یعنی: هر وزن مستقلاً از sigmoid می‌گذرد، سپس میانگین وزن‌دار گرفته می‌شود.
        # اکنون: sigmoid برای هر وزن، Mean برای ترکیب — دقیقاً Eq.4.
        w1 = torch.sigmoid(self.w1)  # sigmoid مطابق Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        # FIX (3-class adaptation): حذف /3 — تقسیم بر 3 logit‌ها را scale می‌کرد و
        # gradient را 3 برابر کوچک می‌کرد (gradient از w*logit → gradient/3)، که
        # یادگیری را کند و ناپایدار می‌کرد. روی argmax بی‌اثر بود اما روی loss
        # مصنوعاً مقدار را بالا نگه می‌داشت (آزمون: loss بدون /3 = 0.817 ، با /3 = 0.933).
        # weighted sum مستقیم هم از نظر ریاضی صحیح‌تر است: CrossEntropyLoss روی
        # logit کار می‌کند و نیازی به normalization قبلی ندارد.
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out  # FIX: /3 حذف شد

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]  # مقاله: Adam بدون weight decay (مقدار پیش‌فرض = 0)

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

# TWO-STAGE TRAINING (paper Sec. 3.6):
# Stage 1: train on 80% of train set, early-stop on 20% val, patience=50, max=300 epochs.
#          Best checkpoint (highest val acc) is restored at end of stage 1.
# Stage 2: retrain on FULL train set (no val split), stop when train loss drops below
#          the best stage-1 train loss achieved, max=200 epochs.
STAGE1_MAX_EPOCHS = 300   # paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # paper says patience 50
STAGE2_MAX_EPOCHS = 200   # paper Sec. 3.6 says max 200 for stage 2

# FIX (مقاله Sec. 3.6 — Paper-Compliant Fix 3/3: Window-level 4:1 val split):
# قبلاً: subject-level split — 2 subject کامل برای val نگه داشته می‌شد.
# مقاله Sec. 3.6: "the training set was randomly split into a first-stage training subset
# and a validation set in a 4:1 ratio."
# این split روی پنجره‌های زمانی (windows) است، نه روی subjects.
# اکنون: TRAIN_SUBS کامل برای stage-1 استفاده می‌شود، اما 20% از windows به صورت
# تصادفی به عنوان val جدا می‌شوند — دقیقاً مطابق روش مقاله.
# نکته: S1_VAL_SUBS و S1_TRAIN_SUBS دیگر استفاده نمی‌شوند؛ split درون run_training
# با SubsetRandomSampler روی indices انجام می‌شود.
# این تغییر باعث می‌شود val acc از windows همان subjects train بیاید (مثل مقاله).


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device, test_session=3):  # CROSS-SESSION: test_session = 1، 2 یا 3
    train_sessions = [s for s in [1, 2, 3] if s != test_session]  # CROSS-SESSION: دو session باقیمانده
    print("\n" + "#" * 70)
    print(f"# CROSS-SESSION: train on sessions {train_sessions}  |  test on session {test_session}")
    print("#" * 70)

    from torch.utils.data import SubsetRandomSampler

    # CROSS-SESSION: همه 26 subject، اما فقط windows مربوط به train_sessions یا test_session
    train_dataset = MMapHybridDataset(ALL_SUBJECTS, "Train_Set",
                                       session_filter=train_sessions,
                                       min_windows_threshold=200)   # CROSS-SESSION
    test_dataset  = MMapHybridDataset(ALL_SUBJECTS, "Test_Set",
                                       session_filter=[test_session],
                                       min_windows_threshold=100)   # CROSS-SESSION

    # FIX: window-level 4:1 split روی TRAIN_SUBS — مطابق مقاله Sec. 3.6
    n_total_train = len(train_dataset)
    indices = list(range(n_total_train))
    random.shuffle(indices)  # seed=42 قبلاً set شده — reproducible
    split = int(n_total_train * 0.8)  # 80% train, 20% val — نسبت 4:1 مقاله
    s1_train_indices = indices[:split]
    s1_val_indices   = indices[split:]
    print(f"[Cross-Session] Window-level 4:1 split: "
          f"{len(s1_train_indices)} train windows, {len(s1_val_indices)} val windows "
          f"(from {len(ALL_SUBJECTS)} subjects, sessions {train_sessions}, ratio={len(s1_train_indices)/len(s1_val_indices):.1f}:1)")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 80% windows
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )  # FIX: SubsetRandomSampler برای 20% windows
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # CROSS-SESSION optimizer: مقاله Sec. 3.6 از Adam با lr=0.001 استفاده می‌کند (بدون weight_decay)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # FIX (3-class adaptation): class-weighted CrossEntropyLoss برای n-back imbalance
    # مقاله روی تکالیف binary (balanced) کار می‌کرد — label_smoothing به تنهایی کافی بود.
    # در n-back سه‌کلاسه، 0-back معمولاً windows بیشتری دارد (آسان‌تر، طولانی‌تر)،
    # که بدون تصحیح باعث می‌شود model به سمت کلاس majority bias پیدا کند و
    # 2-back و 3-back را کمتر یاد بگیرد — دقیقاً مشکلی که در BCI cross-subject
    # classification گزارش شده (Lotte et al., 2018).
    # راه‌حل: inverse-frequency weighting — کلاس‌های کم‌نمونه‌تر وزن بیشتری می‌گیرند.
    # وزن‌ها از train_dataset.y محاسبه می‌شوند (نه test) تا از data leakage جلوگیری شود.
    # label_smoothing=0.1 همچنان حفظ می‌شود — هر دو با هم کار می‌کنند:
    #   class_weight: bias مدل به کلاس majority را اصلاح می‌کند
    #   label_smoothing: overconfidence را کاهش می‌دهد
    # منبع: King & Zeng (2001) — inverse frequency weighting برای imbalanced classification
    _train_labels   = train_dataset.y                                        # numpy array از همه train windows
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float) # [n_0back, n_2back, n_3back]
    _class_weights  = 1.0 / (_class_counts + 1e-6)                          # inverse frequency
    _class_weights  = _class_weights / _class_weights.sum() * 3             # normalize: mean weight = 1
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    # label_smoothing=0.1: Müller et al. (2019) NeurIPS "When Does Label Smoothing Help?"
    # در cross-subject، train/test distribution فرق دارند — label smoothing calibration را بهبود می‌دهد
    # و مانع از overconfident predictions می‌شود که مستقیماً به overfitting منجر می‌شود.
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor, label_smoothing=0.1)  # FIX: class weights اضافه شد

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{test_session}.pth')

    # ------------------------------------------------------------------ #
    # STAGE 1: train on 80% of TRAIN windows, early-stop on 20% val      #
    # مطابق مقاله Sec. 3.6: window-level 4:1 split از train sessions      #
    # ------------------------------------------------------------------ #
    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of train sessions {train_sessions}), "
          f"val on {len(s1_val_indices)} windows (20%) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | window-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')  # loss در epoch بهترین val
    stage1_final_train_loss    = float('inf')  # loss آخرین epoch — fallback وقتی best val در epoch 1 است
    train_acc = 0.0

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss  # هر epoch آپدیت می‌شود

        # stage-1 validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    # Restore best stage-1 weights before stage 2
    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(ALL_SUBJECTS)} subjects, sessions {train_sessions}) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    # Fresh optimizer for stage 2 — همان Adam lr=0.001 مطابق مقاله
    optimizer2 = optim.Adam(model.parameters(), lr=0.001)
    scheduler2  = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=5)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)
        scheduler2.step(avg_s2_loss)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        # Reached max epochs without meeting loss criterion -- save anyway
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (test_session={test_session})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    # FIX: learned weights اکنون از sigmoid می‌آیند (نه softmax)
    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap = (train_acc * 100) - test_acc
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_session{test_session}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE — Cross-Session Hold-Out (مطابق مقاله TSFNet Sec. 3.6)
# سه بار اجرا می‌شود: هر بار یک session به عنوان test، دو session به عنوان train.
# نتیجه نهایی = میانگین test accuracy سه run.
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Cross-Session Hold-Out: 3 runs (each session as test once)")
    print(f"All {len(ALL_SUBJECTS)} subjects used in each run")

    all_session_results = []
    for test_session in [1, 2, 3]:
        result = run_training(weight_decay=0.0, device=device, test_session=test_session)
        all_session_results.append(result)

    print("\n" + "=" * 70)
    print("CROSS-SESSION HOLD-OUT SUMMARY")
    print("=" * 70)
    print(f"{'test_session':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'kappa':>8}")
    for r in all_session_results:
        print(f"{r['test_session']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['kappa_final']:>8.4f}")

    final_acc   = np.mean([r['test_accuracy_final'] for r in all_session_results])
    final_kappa = np.mean([r['kappa_final']          for r in all_session_results])
    print(f"\nFinal average accuracy (3 sessions): {final_acc:.2f}%")
    print(f"Final average Kappa   (3 sessions): {final_kappa:.4f}")

    summary = {
        'session_results': all_session_results,
        'final_average_accuracy': final_acc,
        'final_average_kappa':   final_kappa,
    }
    summary_path = os.path.join(PROCESSED_DATA_PATH, 'cross_session_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\nFull cross-session summary saved to: {summary_path}")
    print("\nDone.")

Running on Device: cuda
Cross-Session Hold-Out: 3 runs (each session as test once)
All 26 subjects used in each run

######################################################################
# CROSS-SESSION: train on sessions [2, 3]  |  test on session 1
######################################################################
Scanning Train_Set - 26 subjects (sessions [2, 3])...
VP001 verified: 682 windows
VP002 verified: 684 windows
VP003 verified: 684 windows
VP004 verified: 501 windows
VP005 verified: 677 windows
VP006 verified: 684 windows
VP007 verified: 684 windows
VP008 verified: 682 windows
VP009 verified: 683 windows
VP010 verified: 682 windows
VP011 verified: 682 windows
VP012 verified: 684 windows
VP013 verified: 682 windows
VP014 verified: 677 windows
VP015 verified: 673 windows
VP016 verified: 684 windows
VP017 verified: 684 windows
VP018 verified: 684 windows
VP019 verified: 684 windows
VP020 verified: 684 windows
VP021 verified: 678 windows
VP022 verified: 684 windows
VP023 v